In [1]:
import pandas as pd
import numpy as np
import gzip
import matplotlib.pyplot as plt
import pickle
from scipy.stats import linregress
import os
import sys
from datetime import date
from sklearn.metrics import auc, f1_score
import random
from sklearn.decomposition import NMF, non_negative_factorization
import seaborn as sns
import time
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.axes_grid1 import axes_grid
import pyreadr
import statistics as stats

In [2]:
#load npz file
mixture_all_cycles = []

#original mixture
fn = gzip.GzipFile('/home/ctrader/Hold/DHS/2018-06-08NC16_NNDSVD_Mixture.npy.gz', "r")
mixture_all_cycles.append(np.float32(np.load(fn)))

for i in range(1, 2):
    print(i)
    filename = '/home/ctrader/Hold/NMF/DHS/fill_in_multiple_at_a_time_saves/reconstructed_threshold_flip_booldhsarr_basis_mixture_to0thresh0.0_to1thresh1.0_cycle'+str(i)+'_CT20220323.npz'
    cycle_np_file = np.load(filename)
    mixture_all_cycles.append(cycle_np_file['new_decomp_mixture'])

mixture_all_cycles = np.array(mixture_all_cycles)

1


In [3]:
n_components, n_DHSs = mixture_all_cycles[0].shape

In [6]:
mixture_all_cycles.shape

(2, 16, 3591898)

In [26]:
DHS_metadata = pd.read_table('/home/ctrader/Hold/DHS/DHS_Index_and_Vocabulary_hg38_WM20190703.txt.gz', header=None)

col_names = DHS_metadata.iloc[0]
DHS_metadata = DHS_metadata[1:]
DHS_metadata.columns = col_names
DHS_metadata = DHS_metadata.reset_index()
DHS_metadata[['identifier']] = DHS_metadata[['identifier']].astype(str)
DHS_metadata = DHS_metadata.drop(columns=['index'])
DHS_metadata['mean_signal'] = DHS_metadata['mean_signal'].astype(float)

#top 10%
DHS_metadata['ind'] = DHS_metadata.index
DHS_metadata_sortby_mean_signal = DHS_metadata.sort_values(['mean_signal'], ascending=False)
top_10_percent_DHSs = DHS_metadata_sortby_mean_signal.iloc[:int(np.round(n_DHSs/10))]
top_10_percent_DHSs_inds = np.array(top_10_percent_DHSs['ind'])
top_10_percent_DHSs_identifiers = np.array(top_10_percent_DHSs['identifier'])

/home/ctrader/.conda/envs/lightfm/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3441: DtypeWarning: Columns (1,2,3,4,5,6,7,8) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [28]:
DHS_metadata.loc[0:1]

,seqname,start,end,identifier,mean_signal,numsamples,summit,core_start,core_end,component,ind
0,chr1,16140,16200,1.10011,0.129388,1,16170,16170,16170,Tissue invariant,0
1,chr1,51868,52040,1.10021,0.080034,1,51970,51970,51970,Placental / trophoblast,1


In [11]:
pickle_in = open('/home/ctrader/Hold/NMF/DHS/gene_DHSs_all_dict_CT20220324.pickle', 'rb')
gene_DHSs_all_dict=pickle.load(pickle_in)

In [20]:
comps_for_heatmap = ['Tissue invariant','Stromal A', 'Primitive / embryonic', 'Stromal B', 'Lymphoid', 'Renal / cancer', 'Placental / trophoblast', 'Neural', 'Cardiac', 'Organ devel. / renal', 'Pulmonary devel.', 'Musculoskeletal', 'Digestive', 'Vascular / endothelial', 'Myeloid / erythroid', 'Cancer / epithelial']
comps_for_heatmap = np.array(comps_for_heatmap)

**Gene Enrichment**

In [4]:
n_DHSs

3591898

In [5]:
#top comps
mixture_all_cycles_top_comp = np.argmax(mixture_all_cycles, axis=1)
mixture_all_cycles_top_comp_val = np.max(mixture_all_cycles, axis=1)

#sum
mixture_all_cycles_sum = np.sum(mixture_all_cycles, axis=1)

In [6]:
#expected enrichment
mixture_cycle0_enrichment =[]
mixture_cycle1_enrichment =[]
for i in range(16):
    mixture_cycle0_enrichment.append(len(np.where(mixture_all_cycles_top_comp[0, :]==i)[0])/n_DHSs)
    mixture_cycle1_enrichment.append(len(np.where(mixture_all_cycles_top_comp[1, :]==i)[0])/n_DHSs)
mixture_cycle0_enrichment = np.array(mixture_cycle0_enrichment)
mixture_cycle1_enrichment = np.array(mixture_cycle1_enrichment)

In [12]:
genes = np.array(list(gene_DHSs_all_dict.keys()))

min_DHS_per_gene = []
max_DHS_per_gene = []
for gene in genes:
    gene_arr = gene_DHSs_all_dict[gene]
    if len(gene_arr)>0:
        min_DHS_per_gene.append(np.min(gene_arr))
        max_DHS_per_gene.append(np.max(gene_arr))
    else:
        min_DHS_per_gene.append(np.nan)
        max_DHS_per_gene.append(np.nan)
min_DHS_per_gene = np.array(min_DHS_per_gene)
max_DHS_per_gene = np.array(max_DHS_per_gene)

In [13]:
comp_differences = mixture_all_cycles_top_comp[0] - mixture_all_cycles_top_comp[1]
DHS_with_comp_differences = np.where(np.abs(comp_differences)>0)[0]

In [14]:
DHS_with_comp_differences.shape

(5767,)

In [15]:
changed_comp_count_per_gene ={}
for gene in genes:
    changed_comp_count_per_gene[gene]=0
    
for DHS in DHS_with_comp_differences:
    if len(genes[(min_DHS_per_gene <= DHS) & (max_DHS_per_gene >= DHS)]) > 0:
        gene_list = genes[(min_DHS_per_gene <= DHS) & (max_DHS_per_gene >= DHS)]
        for gene in gene_list:
            changed_comp_count_per_gene[gene]+=1

In [16]:
changed_comp_count_per_gene_values = np.array(list(changed_comp_count_per_gene.values()))
changed_comp_count_per_gene_keys = np.array(list(changed_comp_count_per_gene.keys()))

In [17]:
top_changed_genes = changed_comp_count_per_gene_keys[np.argsort(-changed_comp_count_per_gene_values)]

In [18]:
#number of genes where there are more counts of enriched DHS after flipping
count = 0
for g in top_changed_genes:
    if len(gene_DHSs_all_dict[g]) > 0:
        original_mode = stats.mode(mixture_all_cycles_top_comp[0, gene_DHSs_all_dict[g]])
        new_mode = stats.mode(mixture_all_cycles_top_comp[1, gene_DHSs_all_dict[g]])
        if original_mode==new_mode:
            original_mode_len = len(np.where(mixture_all_cycles_top_comp[0, gene_DHSs_all_dict[g]]==original_mode)[0])
            new_mode_len = len(np.where(mixture_all_cycles_top_comp[1, gene_DHSs_all_dict[g]]==original_mode)[0])
            if new_mode_len > original_mode_len:
                count+=1
print(count)

668


In [45]:
for g in top_changed_genes[:20]:
    original_mode = stats.mode(mixture_all_cycles_top_comp[0, gene_DHSs_all_dict[g]])
    new_mode = stats.mode(mixture_all_cycles_top_comp[1, gene_DHSs_all_dict[g]])
    if original_mode==new_mode:
        print(g)
        print(comps_for_heatmap[original_mode])
        original_num_of_mode = len(np.where(mixture_all_cycles_top_comp[0, gene_DHSs_all_dict[g]]==original_mode)[0])
        new_num_of_mode = len(np.where(mixture_all_cycles_top_comp[1, gene_DHSs_all_dict[g]]==original_mode)[0])
        print(original_num_of_mode)
        print(new_num_of_mode)
        print(np.log2((original_num_of_mode/len(gene_DHSs_all_dict[g])) / mixture_cycle0_enrichment[original_mode]))
        print(np.log2((new_num_of_mode/len(gene_DHSs_all_dict[g])) / mixture_cycle1_enrichment[original_mode]))
        print()

AC010091.1
Stromal B
70
70
1.2286038752640487
1.2306006705222918

ITPR1
Primitive / embryonic
112
117
-0.01989325477993866
0.041668912060865156

`RP11-36B6.2`
Stromal B
160
161
1.0895545514724212
1.100540129957919

EYS
Neural
320
321
0.6861509436926008
0.6901334719434333

ZNF438
Stromal B
63
63
0.9565243298272479
0.958521125085491

PARK2
Neural
340
339
0.6956112729416672
0.690842935956201

CUL9
Primitive / embryonic
43
49
1.3308204669149823
1.5178179256430961

`RP11-9J18.1`
Stromal B
274
272
0.6669429232303826
0.6583704767784386

FEZ2
Vascular / endothelial
19
19
3.0670610518612875
3.0651234759110695

ARMC4
Neural
23
23
0.09165713822790152
0.09113827424379348

C10orf11
Primitive / embryonic
252
253
-0.21770003783811354
-0.21343401732871647

SGIP1
Stromal B
57
57
0.5940421365892508
0.5960389318474938

`LINC-PINT`
Lymphoid
83
83
1.5714823663251267
1.5708234510175372

STC1
Stromal B
10
10
0.9792444063273313
0.9812412015855745

DLG2
Neural
462
462
0.7541699236065305
0.7536510596224223

MAS

In [43]:
for i in range(20):
    g = top_changed_genes[i]
    original_mode = stats.mode(mixture_all_cycles_top_comp[0, gene_DHSs_all_dict[g]])
    new_mode = stats.mode(mixture_all_cycles_top_comp[1, gene_DHSs_all_dict[g]])
    print(g)
    print(comps_for_heatmap[original_mode])
    print(comps_for_heatmap[new_mode])
    print()
    print((len(np.where(mixture_all_cycles_top_comp[0, gene_DHSs_all_dict[g]]==original_mode)[0])/len(gene_DHSs_all_dict[g])) / mixture_cycle0_enrichment[original_mode])
    print((len(np.where(mixture_all_cycles_top_comp[0, gene_DHSs_all_dict[g]]==new_mode)[0])/len(gene_DHSs_all_dict[g])) / mixture_cycle0_enrichment[new_mode])
    print()
    print((len(np.where(mixture_all_cycles_top_comp[1, gene_DHSs_all_dict[g]]==original_mode)[0])/len(gene_DHSs_all_dict[g])) / mixture_cycle0_enrichment[original_mode])
    print((len(np.where(mixture_all_cycles_top_comp[1, gene_DHSs_all_dict[g]]==new_mode)[0])/len(gene_DHSs_all_dict[g])) / mixture_cycle0_enrichment[new_mode])
    print()
    print()

RADIL
Organ devel. / renal
Primitive / embryonic

6.3717474573456645
1.2892171416223146

4.967803102337299
1.645809116964657


AC010091.1
Stromal B
Stromal B

2.343401044951304
2.343401044951304

2.343401044951304
2.343401044951304


ITPR1
Primitive / embryonic
Primitive / embryonic

0.9863056786970992
0.9863056786970992

1.0303371822103624
1.0303371822103624


`RP11-36B6.2`
Stromal B
Stromal B

2.12808319353042
2.12808319353042

2.1413837134899847
2.1413837134899847


EYS
Neural
Neural

1.6089850750637824
1.6089850750637824

1.6140131534233568
1.6140131534233568


ZNF438
Stromal B
Stromal B

1.9406289903502987
1.9406289903502987

1.9406289903502987
1.9406289903502987


PARK2
Neural
Neural

1.6195705031892023
1.6195705031892023

1.6148070605327633
1.6148070605327633


CUL9
Primitive / embryonic
Primitive / embryonic

2.5154568935648594
2.5154568935648594

2.866450878713444
2.866450878713444


`RP11-9J18.1`
Stromal B
Stromal B

1.58770504687799
1.58770504687799

1.5761159589445741
1.576

In [57]:
genes = np.array(list(gene_DHSs_all_dict.keys()))

In [61]:
mixture_all_cycles_top_comp.shape

(2, 3591898)

In [106]:
#finding the counts of each component in each gene
original_comp_counts_for_each_gene = []
for gene in genes:
    dhss_in_gene = gene_DHSs_all_dict[gene]
    comps_in_gene = mixture_all_cycles_top_comp[0, dhss_in_gene]
    gene_comp_counts = []
    for comp in range(16):
        gene_comp_counts.append(len(np.where(comps_in_gene==comp)[0]))
    original_comp_counts_for_each_gene.append(gene_comp_counts)
original_comp_counts_for_each_gene = np.array(original_comp_counts_for_each_gene)

new_comp_counts_for_each_gene = []
for gene in genes:
    dhss_in_gene = gene_DHSs_all_dict[gene]
    comps_in_gene = mixture_all_cycles_top_comp[1, dhss_in_gene]
    gene_comp_counts = []
    for comp in range(16):
        gene_comp_counts.append(len(np.where(comps_in_gene==comp)[0]))
    new_comp_counts_for_each_gene.append(gene_comp_counts)
new_comp_counts_for_each_gene = np.array(new_comp_counts_for_each_gene)

# comp_counts_for_each_gene = np.array([original_comp_counts_for_each_gene, new_comp_counts_for_each_gene])

DHS_nums_in_each_gene = []
for gene in genes:
    DHS_nums_in_each_gene.append(len(gene_DHSs_all_dict[gene]))
DHS_nums_in_each_gene = np.array(DHS_nums_in_each_gene)

#enrichment of each gene
original_comp_enrichment_for_each_gene = np.log2((original_comp_counts_for_each_gene.T/DHS_nums_in_each_gene).T/mixture_cycle0_enrichment)
new_comp_enrichment_for_each_gene = np.log2((new_comp_counts_for_each_gene.T/DHS_nums_in_each_gene).T/mixture_cycle1_enrichment)

comp_enrichment_for_each_gene = np.array([original_comp_enrichment_for_each_gene, new_comp_enrichment_for_each_gene])

/tmp/ipykernel_87110/345179964.py:30: RuntimeWarning: invalid value encountered in true_divide
  original_comp_enrichment_for_each_gene = np.log2((original_comp_counts_for_each_gene.T/DHS_nums_in_each_gene).T/mixture_cycle0_enrichment)
/tmp/ipykernel_87110/345179964.py:30: RuntimeWarning: divide by zero encountered in log2
  original_comp_enrichment_for_each_gene = np.log2((original_comp_counts_for_each_gene.T/DHS_nums_in_each_gene).T/mixture_cycle0_enrichment)
/tmp/ipykernel_87110/345179964.py:31: RuntimeWarning: invalid value encountered in true_divide
  new_comp_enrichment_for_each_gene = np.log2((new_comp_counts_for_each_gene.T/DHS_nums_in_each_gene).T/mixture_cycle1_enrichment)
/tmp/ipykernel_87110/345179964.py:31: RuntimeWarning: divide by zero encountered in log2
  new_comp_enrichment_for_each_gene = np.log2((new_comp_counts_for_each_gene.T/DHS_nums_in_each_gene).T/mixture_cycle1_enrichment)


In [115]:
comp_enrichment_for_each_gene.shape

(2, 56832, 16)

In [137]:
comp_enrichment_difference_from_original = []
for gene_ind in range(len(comp_enrichment_for_each_gene[0, :, 0])):
    original_max_comp = np.argmax(comp_enrichment_for_each_gene[0, gene_ind, :])
    original_max_comp_val = comp_enrichment_for_each_gene[0, gene_ind, original_max_comp]
    new_max_comp_val = comp_enrichment_for_each_gene[1, gene_ind, original_max_comp]
    comp_enrichment_difference_from_original.append(new_max_comp_val - original_max_comp_val)
    
comp_enrichment_difference_from_original = np.array(comp_enrichment_difference_from_original)

In [142]:
np.nanargmax(comp_enrichment_difference_from_original)

16253

In [143]:
comp_enrichment_for_each_gene[:, 16253, :]

array([[ 1.1878382 ,        -inf,  0.19733746,  0.82724131,  0.35833062,
               -inf,  0.43886291, -0.36151549,        -inf,        -inf,
               -inf,        -inf,  1.31187091,        -inf,        -inf,
         0.93026605],
       [ 1.18453879,        -inf,  0.19588983,  0.82923811,  0.3576717 ,
               -inf,        -inf, -0.36203435,        -inf,        -inf,
               -inf,        -inf,  2.31179114,        -inf,        -inf,
         0.93211951]])

In [145]:
genes[16253]

'`RP11-103F13.1`'